# Two-Stage Hybrid Rocket — Parallel Staging, Planar 3-DOF

**Both engines ignite at t=0 (parallel staging).** S1 decouples after burnout.

```
t=0 ─► RAIL (T1+T2) ─► PARALLEL BURN (T1+T2) ─► [S1 decouple] ─► S2 BURN ─► S2 COAST
         s=L_rail            t=t_burn1              t_burn2         vz<0 → apogee
```

| Parameter | Stage 1 | Stage 2 |
|-----------|---------|----------|
| Total mass | 73 kg (pad) | 58 kg (at separation) |
| Thrust | 1200 N | 400 N |
| Burn time | 8 s | configurable |

- Rail: 7 m, 40° elevation
- Both engines fire from liftoff

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmartos-br/hybrid-rocket-trajectory/blob/main/two_stage_simulation.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3
})

## 1. Configuration Parameters

**PARALLEL STAGING**: Both S1 and S2 engines fire from t=0.  
After S1 burnout, S1 structure decouples. S2 continues alone.

In [ ]:
# ======================================================================
#  LAUNCH SITE & RAIL
# ======================================================================
g0          = 9.81          # gravitational acceleration [m/s^2]
rho         = 1.225         # air density at sea level [kg/m^3]
L_rail      = 7.0           # launch rail length [m]
theta_rail  = np.radians(40)  # rail elevation from horizontal [rad] (40 deg)

# ======================================================================
#  STAGE 1  (booster -- fires from t=0, decouples after burnout)
# ======================================================================
m_total_pad = 73.0          # total pad mass (S1 + S2) [kg]
m_s2_total  = 58.0          # S2 total mass at separation [kg]
m_prop1     = 5.0           # S1 propellant mass [kg]
T1          = 1200.0        # S1 thrust (constant) [N]
t_burn1     = 8.0           # S1 burn time [s]

# Derived
m_s1_total  = m_total_pad - m_s2_total   # 15 kg (S1 structure + S1 propellant)
m_s1_dry    = m_s1_total - m_prop1        # 10 kg (S1 structure only)

# ======================================================================
#  STAGE 2  (sustainer -- ALSO fires from t=0, continues after S1 sep)
# ======================================================================
m_prop2     = 4.0           # S2 propellant mass [kg]
m_s2_dry    = m_s2_total - m_prop2        # S2 dry mass [kg]
T2          = 400.0         # S2 thrust (constant) [N]
t_burn2     = 12.0          # S2 TOTAL burn time from t=0 [s]
                            # (so S2 burns solo for t_burn2 - t_burn1 = 4s after S1 sep)

# ======================================================================
#  AERODYNAMICS
# ======================================================================
d_ref       = 0.15          # reference diameter [m]
S_ref       = np.pi * (d_ref / 2)**2
CD          = 0.45          # axial drag coefficient
CNa         = 3.5           # normal force slope [1/rad]
SM          = 1.5           # static margin [calibers]
Cmq         = -8.0          # pitch damping coefficient

# ======================================================================
#  INERTIA (estimates -- replace with CAD values)
# ======================================================================
L_rocket    = 2.5           # total rocket length [m]
I_s1        = 0.5 * m_s1_dry * (d_ref/2)**2 + m_s1_dry * (L_rocket * 0.3)**2
I_s2        = 0.5 * m_s2_dry * (d_ref/2)**2 + m_s2_dry * (L_rocket * 0.2)**2

# ======================================================================
#  SIMULATION
# ======================================================================
dt          = 0.001         # rail time step [s]
dt_sim      = 0.005         # free-flight time step [s]
t_max       = 300.0         # max simulation time [s]

print("=" * 50)
print("  PARALLEL STAGING CONFIGURATION")
print("=" * 50)
print(f"  Pad mass:        {m_total_pad:.1f} kg")
print(f"  S1 dry:          {m_s1_dry:.1f} kg  |  S1 prop: {m_prop1:.1f} kg")
print(f"  S2 dry:          {m_s2_dry:.1f} kg  |  S2 prop: {m_prop2:.1f} kg")
print(f"  S2 mass at sep:  {m_s2_total:.1f} kg")
print(f"  T1 + T2:         {T1 + T2:.0f} N (combined at liftoff)")
print(f"  T/W at pad:      {(T1 + T2) / (m_total_pad * g0):.2f}")
print(f"  T/W S2 solo:     {T2 / (m_s2_total * g0):.2f}")
print(f"  S1 burns:        0 - {t_burn1:.0f} s")
print(f"  S2 burns:        0 - {t_burn2:.0f} s")
print(f"  S2 solo burn:    {t_burn1:.0f} - {t_burn2:.0f} s  ({t_burn2 - t_burn1:.0f}s after sep)")
print(f"  Rail:            {L_rail:.0f} m at {np.degrees(theta_rail):.0f} deg")
print("=" * 50)

## 2. Physics Engine — Parallel Staging Model

In [ ]:
def prop_frac(t_elapsed, t_burn):
    """Linear propellant depletion: f = max(0, 1 - t/t_burn)"""
    return max(0.0, 1.0 - t_elapsed / t_burn)


def get_phase(t):
    """Phase logic for parallel staging."""
    s1_on = t < t_burn1
    s2_on = t < t_burn2
    if s1_on and s2_on:
        return 'PARALLEL'     # both engines firing
    elif not s1_on and s2_on:
        return 'S2_BURN'      # S1 decoupled, S2 still burning
    else:
        return 'S2_COAST'     # both burned out, ballistic


def get_thrust(t):
    """Total thrust at time t. Both engines from t=0."""
    thrust = 0.0
    if t < t_burn1:
        thrust += T1
    if t < t_burn2:
        thrust += T2
    return thrust


def get_mass_and_inertia(t):
    """Mass and inertia with parallel propellant depletion."""
    phase = get_phase(t)

    f1 = prop_frac(t, t_burn1)  # S1 propellant fraction
    f2 = prop_frac(t, t_burn2)  # S2 propellant fraction

    if phase == 'PARALLEL':
        # Full stack: S1 dry + S2 dry + both propellants
        m = m_s1_dry + m_s2_dry + m_prop1 * f1 + m_prop2 * f2
        I = I_s1 + I_s2
    elif phase == 'S2_BURN':
        # S1 decoupled. Only S2 dry + S2 remaining propellant
        m = m_s2_dry + m_prop2 * f2
        I = I_s2
    else:  # S2_COAST
        m = m_s2_dry
        I = I_s2

    return m, I

## 3. Rail Phase (1-DOF, Euler)

Both engines fire during rail travel.  
`a = (T1 + T2 - D - m*g*sin(theta)) / m`

In [ ]:
def simulate_rail():
    s = 0.0
    V = 0.0
    t = 0.0
    max_g = 0.0

    hist = {'t': [], 'x': [], 'z': [], 'V': [], 'a_g': []}

    while s < L_rail and t < t_max:
        m, _ = get_mass_and_inertia(t)
        T = get_thrust(t)   # T1 + T2 during rail
        D = CD * 0.5 * rho * V**2 * S_ref
        a = (T - D - m * g0 * np.sin(theta_rail)) / m

        ag = abs(a) / g0
        max_g = max(max_g, ag)

        hist['t'].append(t)
        hist['x'].append(s * np.cos(theta_rail))
        hist['z'].append(s * np.sin(theta_rail))
        hist['V'].append(V)
        hist['a_g'].append(ag)

        V += a * dt
        V = max(V, 0.0)
        s += V * dt
        t += dt

    exit_state = {
        't': t,
        'x': s * np.cos(theta_rail),
        'z': s * np.sin(theta_rail),
        'vx': V * np.cos(theta_rail),
        'vz': V * np.sin(theta_rail),
        'theta': theta_rail,
        'omega': 0.0,
        'V': V,
        'q': 0.5 * rho * V**2,
        'max_g': max_g,
    }

    print(f"=== RAIL EXIT ===")
    print(f"  t       = {t:.3f} s")
    print(f"  V       = {V:.2f} m/s  ({V*3.6:.1f} km/h)")
    print(f"  thrust  = {get_thrust(t):.0f} N (T1+T2)")
    print(f"  q       = {exit_state['q']:.1f} Pa")
    print(f"  alt     = {exit_state['z']:.2f} m")
    print(f"  max g   = {max_g:.2f} g")

    return exit_state, hist

## 4. Free Flight (3-DOF, RK4)

State vector: `y = [x, z, vx, vz, theta, omega]`

In [ ]:
def derivatives(t, y):
    x, z, vx, vz, theta, omega = y

    m, I = get_mass_and_inertia(t)
    T = get_thrust(t)

    V = np.sqrt(vx**2 + vz**2)
    if V < 1e-6:
        V = 1e-6

    gamma = np.arctan2(vz, vx)
    alpha = theta - gamma
    q_inf = 0.5 * rho * V**2

    D = CD * q_inf * S_ref
    N = CNa * q_inf * S_ref * alpha

    ax = (T * np.cos(theta) - D * (vx / V) - N * np.sin(theta)) / m
    az = (T * np.sin(theta) - D * (vz / V) + N * np.cos(theta) - m * g0) / m

    xcp_xcm = SM * d_ref
    M_restore = -CNa * q_inf * S_ref * xcp_xcm * alpha
    M_damp = -Cmq * q_inf * S_ref * d_ref**2 / (2 * V) * omega
    theta_ddot = (M_restore + M_damp) / I

    return np.array([vx, vz, ax, az, omega, theta_ddot])


def rk4_step(t, y, h):
    k1 = derivatives(t, y)
    k2 = derivatives(t + h/2, y + h/2 * k1)
    k3 = derivatives(t + h/2, y + h/2 * k2)
    k4 = derivatives(t + h, y + h * k3)
    return y + (h / 6) * (k1 + 2*k2 + 2*k3 + k4)

## 5. Run Full Simulation

In [ ]:
# --- Phase 0: Rail ---
rail_exit, rail_hist = simulate_rail()

# --- Initialize free-flight state ---
y = np.array([
    rail_exit['x'], rail_exit['z'],
    rail_exit['vx'], rail_exit['vz'],
    rail_exit['theta'], rail_exit['omega'],
])
t = rail_exit['t']

# Storage arrays
N_rail = len(rail_hist['t'])
hist = {
    't': list(rail_hist['t']),
    'x': list(rail_hist['x']),
    'z': list(rail_hist['z']),
    'V': list(rail_hist['V']),
    'vx': [v * np.cos(theta_rail) for v in rail_hist['V']],
    'vz': [v * np.sin(theta_rail) for v in rail_hist['V']],
    'theta': [np.degrees(theta_rail)] * N_rail,
    'alpha': [0.0] * N_rail,
    'phase': ['RAIL'] * N_rail,
    'accel_g': list(rail_hist['a_g']),
    'mach': [v / 343.0 for v in rail_hist['V']],
    'thrust': [T1 + T2] * N_rail,
    'mass': [m_total_pad] * N_rail,
}

# Event tracking
events = {'rail_exit': rail_exit}
apogee_found = False
prev_vz = y[3]
s1_sep_logged = False
s2_burnout_logged = False

print(f"\nFree-flight from t={t:.3f}s | Phase: {get_phase(t)}")

while t < t_max:
    phase = get_phase(t)
    y = rk4_step(t, y, dt_sim)
    t += dt_sim

    x, z, vx, vz, theta, omega = y
    V = np.sqrt(vx**2 + vz**2)
    gamma = np.arctan2(vz, vx)
    alpha = theta - gamma
    m, _ = get_mass_and_inertia(t)

    # Downsample: store every 10 steps
    if int(t / dt_sim) % 10 == 0:
        hist['t'].append(t)
        hist['x'].append(x)
        hist['z'].append(z)
        hist['V'].append(V)
        hist['vx'].append(vx)
        hist['vz'].append(vz)
        hist['theta'].append(np.degrees(theta))
        hist['alpha'].append(np.degrees(alpha))
        hist['phase'].append(phase)
        hist['mach'].append(V / 343.0)
        hist['thrust'].append(get_thrust(t))
        hist['mass'].append(m)
        dy = derivatives(t, y)
        hist['accel_g'].append(np.sqrt(dy[2]**2 + dy[3]**2) / g0)

    # --- Event: S1 burnout + separation ---
    if not s1_sep_logged and t >= t_burn1:
        s1_sep_logged = True
        events['s1_sep'] = {
            't': t, 'V': V, 'alt': z, 'Mach': V/343,
            'gamma_deg': np.degrees(gamma), 'downrange': x
        }
        print(f"\n=== S1 BURNOUT + SEPARATION (t={t:.2f}s) ===")
        print(f"  V     = {V:.1f} m/s  (Mach {V/343:.2f})")
        print(f"  alt   = {z:.1f} m")
        print(f"  gamma = {np.degrees(gamma):.1f} deg")
        print(f"  Mass drops: {m_total_pad - m_prop1:.0f} kg -> {m:.1f} kg (S1 struct jettisoned)")

    # --- Event: S2 burnout ---
    if not s2_burnout_logged and t >= t_burn2:
        s2_burnout_logged = True
        events['s2_burnout'] = {'t': t, 'V': V, 'alt': z, 'Mach': V/343}
        print(f"\n=== S2 BURNOUT (t={t:.2f}s) ===")
        print(f"  V     = {V:.1f} m/s  (Mach {V/343:.2f})")
        print(f"  alt   = {z:.1f} m")

    # --- Event: Apogee ---
    if not apogee_found and vz < 0 and prev_vz >= 0:
        apogee_found = True
        events['apogee'] = {
            't': t, 'V': V, 'alt': z,
            'downrange': x, 'Mach': V/343
        }
        print(f"\n=== APOGEE (t={t:.2f}s) ===")
        print(f"  altitude  = {z:.1f} m  ({z/1000:.2f} km)")
        print(f"  downrange = {x:.1f} m")
        print(f"  V         = {V:.1f} m/s")

    prev_vz = vz

    # --- Termination: ground impact ---
    if z < 0 and t > 1.0:
        events['impact'] = {'t': t, 'x': x, 'V': V}
        print(f"\n=== GROUND IMPACT (t={t:.2f}s) ===")
        print(f"  downrange = {x:.1f} m  ({x/1000:.2f} km)")
        print(f"  V_impact  = {V:.1f} m/s")
        break

print(f"\nDone. {len(hist['t'])} points.")

## 6. Trajectory Plot

In [ ]:
t_a = np.array(hist['t'])
x_a = np.array(hist['x'])
z_a = np.array(hist['z'])
V_a = np.array(hist['V'])
ph_a = np.array(hist['phase'])

colors = {
    'RAIL':      '#888888',
    'PARALLEL':  '#e74c3c',
    'S2_BURN':   '#2ecc71',
    'S2_COAST':  '#3498db',
}

fig, ax = plt.subplots(figsize=(14, 8))
for name, c in colors.items():
    mask = ph_a == name
    if mask.any():
        ax.scatter(x_a[mask]/1000, z_a[mask]/1000, c=c, s=1.5, label=name, zorder=2)

if 'apogee' in events:
    ap = events['apogee']
    ax.plot(ap['downrange']/1000, ap['alt']/1000, 'r*', ms=15,
            label=f"Apogee: {ap['alt']:.0f} m", zorder=5)
if 's1_sep' in events:
    s = events['s1_sep']
    ax.plot(s['downrange']/1000, s['alt']/1000, 'k^', ms=10,
            label=f"S1 sep: {s['alt']:.0f} m", zorder=5)

ax.set_xlabel('Downrange [km]')
ax.set_ylabel('Altitude [km]')
ax.set_title('Two-Stage Rocket Trajectory \u2014 Parallel Staging (3-DOF)')
ax.legend(loc='upper right', fontsize=9)
ax.set_aspect('equal')
ax.set_ylim(bottom=-0.1)
plt.tight_layout()
plt.show()

## 7. Time History Plots

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

def shade(ax):
    """Shade background by flight phase."""
    regions = [
        (0, rail_exit['t'], '#888888', 'Rail'),
        (rail_exit['t'], t_burn1, '#e74c3c', 'Parallel'),
        (t_burn1, t_burn2, '#2ecc71', 'S2 Burn'),
    ]
    for t0, t1, c, _ in regions:
        ax.axvspan(t0, t1, alpha=0.08, color=c)
    ax.axvline(t_burn1, ls=':', color='black', alpha=0.3, lw=0.8)
    ax.axvline(t_burn2, ls=':', color='black', alpha=0.3, lw=0.8)

# --- Altitude ---
ax = axes[0, 0]
ax.plot(t_a, z_a, 'b-', lw=1.2)
shade(ax)
if 'apogee' in events:
    ax.axhline(events['apogee']['alt'], ls='--', color='red', alpha=0.4, lw=0.8)
    ax.annotate(f"Apogee: {events['apogee']['alt']:.0f} m",
                xy=(events['apogee']['t'], events['apogee']['alt']),
                fontsize=9, color='red')
ax.set_ylabel('Altitude [m]')
ax.set_title('Altitude vs Time')

# --- Velocity ---
ax = axes[0, 1]
ax.plot(t_a, V_a, 'r-', lw=1.2)
shade(ax)
ax.set_ylabel('Velocity [m/s]')
ax.set_title('Velocity vs Time')

# --- Mach ---
ax = axes[1, 0]
ax.plot(t_a, np.array(hist['mach']), 'g-', lw=1.2)
ax.axhline(1.0, ls='--', color='black', alpha=0.4, lw=0.8, label='Mach 1')
shade(ax)
ax.set_ylabel('Mach')
ax.set_title('Mach Number')
ax.legend(fontsize=9)

# --- AoA ---
ax = axes[1, 1]
alpha_a = np.clip(np.array(hist['alpha']), -20, 20)
ax.plot(t_a, alpha_a, 'm-', lw=0.8)
shade(ax)
ax.set_ylabel('AoA [deg]')
ax.set_title('Angle of Attack')

# --- Thrust ---
ax = axes[2, 0]
ax.plot(t_a, np.array(hist['thrust']), 'k-', lw=1.5)
shade(ax)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Thrust [N]')
ax.set_title('Thrust Profile')
ax.set_ylim(bottom=-50)

# --- Mass ---
ax = axes[2, 1]
ax.plot(t_a, np.array(hist['mass']), 'b-', lw=1.5)
shade(ax)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Mass [kg]')
ax.set_title('Mass vs Time (note S1 jettison at sep)')

for a in axes.flat:
    a.set_xlabel('Time [s]')

plt.tight_layout()
plt.show()

## 8. Acceleration Profile

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
accel_a = np.array(hist['accel_g'])
ax.plot(t_a, accel_a, 'k-', lw=1.0)
shade(ax)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Acceleration [g]')
ax.set_title('Acceleration Profile')
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()
print(f"Peak accel: {accel_a.max():.2f} g")

## 9. Summary

In [ ]:
print("=" * 65)
print("  TWO-STAGE PARALLEL ROCKET \u2014 FLIGHT SUMMARY")
print("=" * 65)

re  = events.get('rail_exit', {})
sep = events.get('s1_sep', {})
s2b = events.get('s2_burnout', {})
ap  = events.get('apogee', {})
imp = events.get('impact', {})

rows = [
    ('Rail exit',    f"{re.get('t',0):.2f}", f"{re.get('V',0):.1f}",  f"{re.get('z',0):.1f}"),
    ('S1 sep',       f"{sep.get('t',0):.2f}",f"{sep.get('V',0):.1f}", f"{sep.get('alt',0):.1f}"),
    ('S2 burnout',   f"{s2b.get('t',0):.2f}",f"{s2b.get('V',0):.1f}", f"{s2b.get('alt',0):.1f}"),
    ('Apogee',       f"{ap.get('t',0):.2f}", f"{ap.get('V',0):.1f}",  f"{ap.get('alt',0):.1f}"),
    ('Impact',       f"{imp.get('t',0):.2f}",f"{imp.get('V',0):.1f}", f"{imp.get('x',0):.1f} (range)"),
]

print(f"{'Event':<14} {'Time [s]':<10} {'V [m/s]':<12} {'Alt/Range [m]'}")
print("-" * 65)
for name, ts, vs, alt in rows:
    print(f"{name:<14} {ts:<10} {vs:<12} {alt}")

print("\n" + "=" * 65)
if ap:
    print(f"  \u2605 APOGEE:    {ap['alt']:.0f} m ({ap['alt']/1000:.2f} km)")
    print(f"  \u2605 DOWNRANGE: {ap.get('downrange',0):.0f} m at apogee")
if imp:
    print(f"  \u2605 RANGE:     {imp['x']:.0f} m ({imp['x']/1000:.2f} km)")
    print(f"  \u2605 FLIGHT:    {imp['t']:.1f} s")
print("=" * 65)

---

### Model Notes

**What changed from serial staging:**
- Both S1 and S2 engines ignite at t=0 (combined 1600 N at liftoff)
- During parallel burn, both propellants deplete simultaneously
- At S1 burnout (t=8s), S1 dry structure is jettisoned → sudden mass drop
- S2 continues burning solo until t_burn2, then coasts to apogee

**Current simplifications:**
1. Constant CD/CN\u03b1 (no Mach table)
2. Uniform atmosphere (\u03c1 = const)
3. Linear propellant depletion
4. Planar only (no wind/yaw/roll)

**Parameters to tune:**
- `m_prop1`, `m_prop2` — propellant masses
- `t_burn2` — S2 total burn time (currently 12s, so 4s solo after sep)
- `d_ref`, `CD`, `CN\u03b1`, `SM` — from OpenRocket/RASAero